In [1]:
import os

import kaggle_evaluation.aimo_3_inference_server
import pandas as pd
import polars as pl


class Model:
    """A dummy model."""

    def __init__(self):
        self._model = None

    def load(self):
        """Simulate model loading."""
        print("Loading model...")
        # Just return a "model" that always answers with 0
        return lambda problem: 0

    def predict(self, problem: str):
        # Employ lazy loading: load model on the first model.predict call
        if self._model is None:
            self._model = self.load()
        return self._model(problem)


model = Model()


# Replace this function with your inference code.
# The function should return a single integer between 0 and 99999, inclusive.
def predict(id_: pl.Series, problem: pl.Series) -> pl.DataFrame | pd.DataFrame:
    """Make a prediction."""
    # Unpack values
    id_ = id_.item(0)
    problem_text: str = problem.item(0)
    # Make a prediction
    # The model is loaded on the first call
    prediction = model.predict(problem_text)
    return pl.DataFrame({'id': id_, 'answer': prediction})


inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(
    predict
)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # You MUST call this within 15 minutes of the script starting. This is to
    # ensure a "fast fail" in case a bug prevents the inference server from starting.
    # Do anything that might take a long time (like model loading) in the predict
    # function, which has no time limit.
    inference_server.serve()
else:
    inference_server.run_local_gateway(
        ('/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv',)
    )


Loading model...


In [2]:
!pip install transformers accelerate torch

In [3]:
import torch 

assert torch.cuda.is_available(), "No GPU detected — enable GPU in Kaggle: Settings → Accelerator → GPU T4 x2"
print(f"GPUs available : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} | "
          f"{torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB")

display(f"VRAM free: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

GPUs available : 2
  GPU 0: Tesla T4 | 15.64 GB
  GPU 1: Tesla T4 | 15.64 GB


'VRAM free: 15.5 GB'

In [57]:
import pandas as pd 

class AIMODataset:
    def __init__(self):
        self.train_df = pd.read_csv("/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv")
        self.test_df = pd.read_csv("/kaggle/input/ai-mathematical-olympiad-progress-prize-3/test.csv")

    @property
    def train_samples(self):
        return self.train_df[['problem', 'answer']].to_markdown(index=False, tablefmt="grid")

    @property
    def test_sampels(self):
        return self.test_df.to_string(index=False)

data = AIMODataset()

sample_idx = 1
sample_row = data.train_df.iloc[sample_idx]
sample_id = sample_row.id
sample_problem = sample_row.problem
sample_answer = sample_row.answer

print(f"Finished retrieving samples (ID-PROB-ANSWER):\nID:\n\t{sample_id}\nQ:\n\t{sample_problem}\n\nA:\n\t{sample_answer}")

Finished retrieving samples (ID-PROB-ANSWER):
ID:
	26de63
Q:
	Define a function $f \colon \mathbb{Z}_{\geq 1} \to \mathbb{Z}_{\geq 1}$ by
\begin{equation*}
    f(n) = \sum_{i = 1}^n \sum_{j = 1}^n j^{1024} \left\lfloor\frac1j + \frac{n-i}{n}\right\rfloor.
\end{equation*}
Let $M=2 \cdot 3 \cdot 5 \cdot 7 \cdot 11 \cdot 13$ and let $N = f{\left(M^{15}\right)} - f{\left(M^{15}-1\right)}$. Let $k$ be the largest non-negative integer such that $2^k$ divides $N$. What is the remainder when $2^k$ is divided by $5^7$?

A:
	32951


In [5]:
# Load the Model with Hugging Face
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

from transformers import AutoModelForCausalLM, AutoTokenizer

# parent model name = "Qwen/Qwen3-30B-A3B"
MODEL_PATH = "/kaggle/input/models/qwen-lm/qwen-3/transformers/4b-base/1"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(
            MODEL_PATH,
            dtype=torch.bfloat16,   
            local_files_only=True,
        )

model.to("cuda:0")

display(f"Completed loading model from path {model_path}")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

'Completed loading model from path /kaggle/input/models/qwen-lm/qwen-3/transformers/4b-base/1'

In [18]:
# Complete Script to load the main model with Hugging Face Transformers
DEFAULT_DECODE_KWARGS = {
    "max_new_tokens": 32768
}

EXAMPLE_PROBLEM = "Let $ABC$ be an acute-angled triangle with integer side lengths and $AB<AC$. Points $D$ and $E$ lie on segments $BC$ and $AC$, respectively, such that $AD=AE=AB$. Line $DE$ intersects $AB$ at $X$. Circles $BXD$ and $CED$ intersect for the second time at $Y \\neq D$. Suppose that $Y$ lies on line $AD$. There is a unique such triangle with minimal perimeter. This triangle has side lengths $a=BC$, $b=CA$, and $c=AB$. Find the remainder when $abc$ is divided by $10^{5}$."
EXAMPLE_ANSWER = "336"

SYS_PROMPT = """Given the mathematical olympiad problem below, generate a mathematical olympiad problem of same mathematical concept but is more {challenge_level} compared to the given problem. Return your response alike the given example input and output below.

## Example User Input
**Problem**
{example_problem}
**Answer**
{example_answer}

## Example Output Response
**Problem**
Contains your response.
**Answer**
Contains the respective answer for the problem.
"""

CHALLENGE_LEVELS = ["easy", "medium", "hard"]

class QwenDataAgent:
    # parent model name = "Qwen/Qwen3-30B-A3B"
    def __init__(self, tokenizer: AutoTokenizer, model: AutoModelForCausalLM):
        self.tokenizer = tokenizer
        self.model = model
        self.decode_kwargs = DEFAULT_DECODE_KWARGS.copy()
        self.sys_prompt = SYS_PROMPT
        self.responses = []

    def system_prompt(self, level: str = "hard"):
        return self.sys_prompt.format(
            example_problem=EXAMPLE_PROBLEM,
            example_answer=EXAMPLE_ANSWER,
            challenge_level=level
        )
        
    def generate(self, id: str | int, problem: str, answer: str, system_prompt_level: str = "hard"):

        try:

            user_input = f"**Problem**\n{problem}\n**Answer**\n{answer}"
            inputs = [
                {"role": "system", "content": self.system_prompt(level=system_prompt_level) },
                {"role": "user", "content": user_input }
            ]

            text = self.tokenizer.apply_chat_template(
                inputs,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
            )
            model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

            print(f"Input tensor device: {model_inputs.input_ids.device}")

            # conduct text completion
            generated_ids = self.model.generate(
                **model_inputs,
                **self.decode_kwargs
            )
            output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

            print(f"Finished generating response IDs: {generated_ids}")
            
            # rindex finding 151668 (</think>)
            try:
                index = len(output_ids) - output_ids[::-1].index(151668)
            except ValueError:
                index = 0
                
            thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
            content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")
            
            #print("thinking content:", thinking_content)
            #print("content:", content)
            response = {
                "id": id,
                "problem": problem,
                "answer": answer,
                "thinking": thinking_content,
                "content": content
            }
            
            self.responses.append(response)
            return response 
            
        except Exception as e:
            raise e

qwe = QwenDataAgent(tokenizer=tokenizer, model=model)

display("Finished configuring models")

'Finished configuring models'

In [58]:
qwe.generate(id=sample_id, problem=sample_problem, answer=sample_answer)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Input tensor device: cuda:0
Finished generating response IDs: tensor([[151644,   8948,    198,  22043,    279,  35972,    297,  13842,  62944,
           3491,   3685,     11,   6923,    264,  35972,    297,  13842,  62944,
           3491,    315,   1852,  35972,   7286,    714,    374,    803,   2588,
           7707,    311,    279,   2661,   3491,     13,   3411,    697,   2033,
          25992,    279,   2661,   3110,   1946,    323,   2550,   3685,    382,
            565,  13383,   2657,   5571,    198,    334,  31198,   1019,  10061,
            400,  25411,      3,    387,    458,  29783,     12,  38940,  21495,
            448,   7546,   3108,  28316,    323,    400,   1867,     27,   1706,
          12947,  20725,    400,     35,      3,    323,    400,     36,      3,
          10246,    389,  20632,    400,   4897,      3,    323,    400,   1706,
          54876,  15576,     11,   1741,    429,    400,   1808,     28,  13669,
             28,   1867,  12947,   7083,    400

{'id': '26de63',
 'problem': 'Define a function $f \\colon \\mathbb{Z}_{\\geq 1} \\to \\mathbb{Z}_{\\geq 1}$ by\n\\begin{equation*}\n    f(n) = \\sum_{i = 1}^n \\sum_{j = 1}^n j^{1024} \\left\\lfloor\\frac1j + \\frac{n-i}{n}\\right\\rfloor.\n\\end{equation*}\nLet $M=2 \\cdot 3 \\cdot 5 \\cdot 7 \\cdot 11 \\cdot 13$ and let $N = f{\\left(M^{15}\\right)} - f{\\left(M^{15}-1\\right)}$. Let $k$ be the largest non-negative integer such that $2^k$ divides $N$. What is the remainder when $2^k$ is divided by $5^7$?',
 'answer': np.int64(32951),
 'thinking': '',
 'content': '**Problem**\nLet $ABC$ be an acute-angled triangle with integer side lengths and $AB<AC$. Points $D$ and $E$ lie on segments $BC$ and $AC$, respectively, such that $AD=AE=AB$. Line $DE$ intersects $AB$ at $X$. Circles $BXD$ and $CED$ intersect for the second time at $Y \\neq D$. Suppose that $Y$ lies on line $AD$. There is a unique such triangle with minimal perimeter. This triangle has side lengths $a=BC$, $b=CA$, and $c=A

In [8]:
# prepare the model input
prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
display(f"Completed preparing model inputs: {model_inputs}")

"Completed preparing model inputs: {'input_ids': tensor([[151644,    872,    198,  35127,    752,    264,   2805,  16800,    311,\n           3460,   4128,   1614,     13, 151645,    198, 151644,  77091,    198]],\n       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],\n       device='cuda:0')}"

In [37]:
#from transformers import GenerationConfig
#?GenerationConfig

Init signature: GenerationConfig(**kwargs)
Docstring:     
Class that holds a configuration for a generation task. A `generate` call supports the following generation methods
for text-decoder, text-to-text, speech-to-text, and vision-to-text models:

    - *greedy decoding* if `num_beams=1` and `do_sample=False`
    - *multinomial sampling* if `num_beams=1` and `do_sample=True`
    - *beam-search decoding* if `num_beams>1` and `do_sample=False`
    - *beam-search multinomial sampling* if `num_beams>1` and `do_sample=True`
    - *assisted decoding* if `assistant_model` or `prompt_lookup_num_tokens` is passed to `.generate()`

To learn more about decoding strategies refer to the [text generation strategies guide](../generation_strategies).

<Tip>

A large number of these flags control the logits or the stopping criteria of the generation. Make sure you check
the [generate-related classes](https://huggingface.co/docs/transformers/internal/generation_utils) for a full
description of the po

In [11]:
# conduct text completion
model.eval()

with torch.inference_mode():
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512,
        do_sample=False,            
        pad_token_id=tokenizer.eos_token_id,
    )

#output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

display(f"Completed Text Completion: \nComplete Output: {generated_ids}")

"Completed Text Completion: \nComplete Output: tensor([[151644,    872,    198,  35127,    752,    264,   2805,  16800,    311,\n           3460,   4128,   1614,     13, 151645,    198, 151644,  77091,    198,\n          34253,   4128,   4119,    320,   4086,  21634,      8,    525,    264,\n            943,    315,  20443,  11229,    320,  15469,      8,    429,    646,\n           6923,   3738,  12681,   1467,   3118,    389,    279,   1946,    807,\n           5258,     13,   2379,    525,  16176,    389,  12767,  14713,    315,\n           1467,    821,     11,   1741,    438,   6467,     11,   9709,     11,\n            323,  13037,     11,    323,    646,   3535,    323,   6923,   1467,\n            304,    264,   8045,    315,  15459,    323,   9222,     13,    444,\n          10994,     82,    525,   1483,    304,    264,   6884,   2088,    315,\n           8357,     11,   2670,   6236,  61905,     11,   4108,  56519,     11,\n            323,   2213,   9471,     13,   2379,   

In [14]:
# parsing thinking content

output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)


thinking content: 
content: Large language models (LLMs) are a type of artificial intelligence (AI) that can generate human-like text based on the input they receive. They are trained on vast amounts of text data, such as books, articles, and websites, and can understand and generate text in a variety of languages and styles. LLMs are used in a wide range of applications, including chatbots, virtual assistants, and content generation. They are becoming increasingly popular as they can provide a more natural and engaging user experience than traditional rule-based systems.
